# Lesson 08 - Multi-Agent Design Pattern


## Setup


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Why Multi-Agent Systems?

Real-world tasks like trip planning involve many different kinds of expertise — logistics, local knowledge, budgeting, and more. One agent wey dey try handle everything go quick become hard to manage.

Multi-agent systems solve dis through **specialization**: each agent dey focus on one area of expertise, dey produce higher-quality results pass one wey sabi everything small. Dem still improve **scalability** — you fit add new agents (e.g., flight specialist, restaurant critic) without to rewrite di workflow wey dey already. The agents join together through one structured pipeline, dey pass context from one go the next.


## Mekim Special Agent Dem


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Building a Sequential Workflow

`WorkflowBuilder` dey allow you connect agents join inside one directed graph. Here we dey create one simple two-step pipeline: the **TravelPlanner** dey draft the itinerary, then the **TravelConcierge** go check am and improve am.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Adding More Agents to the Workflow

One of di biggest advantage of di multi-agent pattern na how e easy to extend. Below we add one **BudgetReviewer** agent wey dey check di plan against di traveler's budget, dey flag items wey fit push costs beyond di limit, and dey suggest money-saving alternatives. Di workflow now dey run three agents one after di oda:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

For dis lesson, you learn how to:

1. **Create specialized agents** — each get one focused role (planning, concierge, budget review).
2. **Wire agents into a sequential workflow** using `WorkflowBuilder` and `add_edge`.
3. **Stream output** from one multi-agent pipeline, dey track which agent dey talk.
4. **Extend a workflow** by adding new agents to the chain without changing the ones wey already dey.

The multi-agent design pattern dey keep each agent simple but e fit produce better, well-checked results pass any single agent fit do alone.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document na AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) wey e use do di translation. Even though we try make am correct, abeg sabi say automatic translation fit get mistake or no too accurate. Di original document wey e dey for im own language na di correct one. If na important information, make you use professional human translation. We no go responsible if you misunderstand or waka body because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
